In [ ]:
import pandas as pd

import gridcast.modeling.train as tr

In [ ]:
# reuse the same data prep as train.py, no need to duplicate it here
df = tr.build_labels(tr.dataset())
df = tr.build_features(df)
df.shape

In [ ]:
# same split logic as train.py, but kept as separate visible variables
test_start = df["time"].max() - pd.DateOffset(months=tr.test_months)
val_start = test_start - pd.DateOffset(months=tr.val_months)

train = tr.split_by_date(df, None, val_start)
val = tr.split_by_date(df, val_start, test_start)
test = tr.split_by_date(df, test_start, None)

print(train["time"].min(), "→", val_start, "→", test_start, "→", df["time"].max())

In [ ]:
import mlflow

mlflow.set_experiment("gridcast-lgbm-notebook")

h = 1
model, feature_cols = tr.train_horizon(train, val, h)
metrics = tr.evaluate(model, test, feature_cols, h)
metrics["train_r2"] = tr.evaluate(model, train, feature_cols, h)["r2"]
metrics["val_r2"] = tr.evaluate(model, val, feature_cols, h)["r2"]

with mlflow.start_run(run_name=f"{h}h_notebook"):
    mlflow.log_params({"horizon_h": h, **model.get_params()})
    mlflow.log_metrics(metrics)

metrics

`tr.train_horizon` hardcodes `LGBMRegressor(objective="regression")` with no other params. To try different hyperparameters, copy its body here and edit `model = lgb.LGBMRegressor(...)` directly, rather than editing `train.py`.

Alternative: instead of retraining, pull an already-registered model from MLflow to inspect/compare against.

In [ ]:
import mlflow.lightgbm

# h = 24
registered_name = f"gridcast-lgbm-{h}h"
registry_model = mlflow.lightgbm.load_model(f"models:/{registered_name}/Staging")

registry_metrics = tr.evaluate(registry_model, test, registry_model.feature_name_, h)
registry_metrics